In [0]:
import numpy as np
import mlflow
import mlflow.tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from mlflow.models import infer_signature

ohlcv = spark.table("workspace.bronze.ohlcv_adj_close").toPandas().set_index("index")

# ── Config ─────────────────────────────────────────────────────────────────────
TICKER      = "2888.HK"
SEQ_LEN     = 30
TEST_RATIO  = 0.2
EPOCHS      = 5
BATCH_SIZE  = 32
MLFLOW_EXP  = "/Users/dong19941207@gmail.com/stock_price_prediction"

# ── 1. Prepare Data ────────────────────────────────────────────────────────────
prices        = ohlcv[TICKER].dropna().values.reshape(-1, 1)
scaler        = MinMaxScaler()
prices_scaled = scaler.fit_transform(prices)

def make_sequences(data: np.ndarray, seq_len: int):
    n       = len(data)
    indices = np.arange(seq_len + 1)[None, :] + np.arange(n - seq_len)[:, None]
    windows = data[indices, 0]
    X       = windows[:, :-1, np.newaxis]
    y       = windows[:, -1:  ]
    return X, y

X, y            = make_sequences(prices_scaled, SEQ_LEN)
split           = int(len(X) * (1 - TEST_RATIO))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ── 2. Build Model ─────────────────────────────────────────────────────────────
def build_model(seq_len: int) -> Sequential:
    model = Sequential([
        LSTM(64, input_shape=(seq_len, 1), return_sequences=False),
        Dense(32, activation="relu"),
        Dense(1),
    ], name="LSTM_stock_predictor")
    model.compile(optimizer="adam", loss="mse")
    return model

# ── 3. Train & Log with MLflow ─────────────────────────────────────────────────
mlflow.set_experiment(MLFLOW_EXP)
model_name = f"StockPredictor_{TICKER.replace('.', '_')}"

with mlflow.start_run(run_name=f"LSTM_{TICKER}") as run:

    # -- log params
    mlflow.log_params({
        "ticker"     : TICKER,
        "seq_len"    : SEQ_LEN,
        "epochs"     : EPOCHS,
        "batch_size" : BATCH_SIZE,
        "lstm_units" : 64,
        "test_ratio" : TEST_RATIO,
        "train_size" : len(X_train),
        "test_size"  : len(X_test),
    })

    # -- train
    model = build_model(SEQ_LEN)
    model.fit(X_train, y_train, validation_split=0.1,
              epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)

    # -- evaluate
    y_pred = scaler.inverse_transform(model.predict(X_test))
    y_true = scaler.inverse_transform(y_test)

    mae     = mean_absolute_error(y_true, y_pred)
    rmse    = np.sqrt(mean_squared_error(y_true, y_pred))
    mape    = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2      = r2_score(y_true, y_pred)
    dir_acc = np.mean(np.sign(np.diff(y_true.flatten())) ==
                      np.sign(np.diff(y_pred.flatten()))) * 100

    mlflow.log_metrics({"MAE": mae, "RMSE": rmse, "MAPE": mape,
                        "R2": r2, "Directional_Accuracy": dir_acc})

    print(f"MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%"
          f"  R2={r2:.4f}  Dir_Acc={dir_acc:.1f}%")

    # -- register model
    signature  = infer_signature(X_train[:5], model.predict(X_train[:5]))
    model_info = mlflow.tensorflow.log_model(
        model,
        name                  = "lstm_model",
        signature             = signature,
        input_example         = X_train[:5],
        registered_model_name = model_name,
    )

    # -- champion / challenger alias
    client      = mlflow.MlflowClient()
    new_version = model_info.registered_model_version

    try:
        champion    = client.get_model_version_by_alias(model_name, "champion")
        champ_run   = client.get_run(champion.run_id)
        champ_mape  = champ_run.data.metrics["MAPE"]
        champ_r2    = champ_run.data.metrics["R2"]

        if mape < champ_mape and r2 > champ_r2:
            client.set_registered_model_alias(model_name, "challenger", champion.version)
            client.set_registered_model_alias(model_name, "champion",   new_version)
            print(f"Version {new_version} is CHAMPION  |  previous v{champion.version} is CHALLENGER")
        else:
            client.set_registered_model_alias(model_name, "challenger", new_version)
            print(f"Version {new_version} is CHALLENGER  |  champion remains v{champion.version}")

    except Exception:
        # no champion yet, crown the first model
        client.set_registered_model_alias(model_name, "champion", new_version)
        print(f"Version {new_version} is CHAMPION (first run)")

    print(f"Run ID : {run.info.run_id}")